In [1]:
!pip install -q pydantic==2.12.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 24.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
gradio 5.38.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.


In [2]:
!pip install -qi https://test.pypi.org/simple/ rwkvx==0.1.0.dev3

In [6]:
!pip install -q rwkv huggingface_hub

In [25]:
import os
import gc

import huggingface_hub
import torch

In [7]:
os.environ["RWKV_V7_ON"] = '1'
os.environ["RWKV_JIT_ON"] = '1'
os.environ["RWKV_CUDA_ON"] = '1' # if '1' then use CUDA kernel for seq mode (much faster)

from rwkv.model import RWKV
from rwkv.utils import PIPELINE, PIPELINE_ARGS

Using /root/.cache/torch_extensions/py311_cu124 as PyTorch extensions root...
Creating extension directory /root/.cache/torch_extensions/py311_cu124/wkv_cuda...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py311_cu124/wkv_cuda/build.ninja...
/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module wkv_cuda...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


[1/4] c++ -MMD -MF gemm_fp16_cublas.o.d -DTORCH_EXTENSION_NAME=wkv_cuda -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/TH -isystem /usr/local/lib/python3.11/dist-packages/torch/include/THC -isystem /usr/local/cuda/include -isystem /usr/include/python3.11 -D_GLIBCXX_USE_CXX11_ABI=0 -fPIC -std=c++17 -c /usr/local/lib/python3.11/dist-packages/rwkv/cuda/gemm_fp16_cublas.cpp -o gemm_fp16_cublas.o 
[2/4] c++ -MMD -MF wrapper.o.d -DTORCH_EXTENSION_NAME=wkv_cuda -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include -isystem /usr/local/lib/python3.11/dist-p

Loading extension module wkv_cuda...
Using /root/.cache/torch_extensions/py311_cu124 as PyTorch extensions root...
Creating extension directory /root/.cache/torch_extensions/py311_cu124/wkv7s...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py311_cu124/wkv7s/build.ninja...
/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module wkv7s...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


[1/3] c++ -MMD -MF rwkv7_op.o.d -DTORCH_EXTENSION_NAME=wkv7s -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/TH -isystem /usr/local/lib/python3.11/dist-packages/torch/include/THC -isystem /usr/local/cuda/include -isystem /usr/include/python3.11 -D_GLIBCXX_USE_CXX11_ABI=0 -fPIC -std=c++17 -c /usr/local/lib/python3.11/dist-packages/rwkv/cuda/rwkv7_op.cpp -o rwkv7_op.o 
[2/3] /usr/local/cuda/bin/nvcc --generate-dependencies-with-compile --dependency-output rwkv7.cuda.o.d -DTORCH_EXTENSION_NAME=wkv7s -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include

Loading extension module wkv7s...


In [10]:
model_title = "rwkv7-g1a4-2.9b-20251118-ctx8192"
# model_title = "rwkv7-g0a4-7.2b-20251208-ctx8192"
# model_title = "rwkv7-g0b-13.3b-20251130-ctx8192"
model_path = huggingface_hub.hf_hub_download(repo_id="BlinkDL/rwkv7-g1", filename=f"{model_title}.pth")
rwkv_model = RWKV(model=model_path.replace('.pth',''), strategy='cuda fp16')
rwkv_pipeline = PIPELINE(rwkv_model, "rwkv_vocab_v20230424")

rwkv7-g1a4-2.9b-20251118-ctx8192.pth:   0%|          | 0.00/5.90G [00:00<?, ?B/s]

Loading /root/.cache/huggingface/hub/models--BlinkDL--rwkv7-g1/snapshots/b25bc6bccda433ca7c63eb571d6b12257784b5cc/rwkv7-g1a4-2.9b-20251118-ctx8192 (cuda fp16)



In [36]:
class RWKVModel:
    """
    A subclass of Model for RWKV7 inference, adapted from the experimental notebook.
    """
    def __init__(
        self,
        model,
        pipeline
    ):    
        # Setup RWKV
        self.CTX_LIMIT = 6000
        self.PENALTY_DECAY = 0.996
        self._model = model
        self._pipeline = pipeline

        # Model state
        self.state = None

    def reset_state(self) -> None:
        self.stare = None
    
    def generate(
        self,
        prompt: str,
    ) -> str:
        """Process the input messages and return the model's response."""
        print(f"Input prompt: ", prompt)

        output = self.run_rwkv(prompt)
        output_text = output.strip()

        print(f"Output: ", output_text)
        
        return output_text

    def run_rwkv(
        self,
        ctx,
        token_count=2000,
        temperature=1.0,
        top_p=0.3,
        presencePenalty=0.5,
        countPenalty=0.5,
    ):    
        ctx = ctx.strip()
        all_tokens = []
        out_last = 0
        out_str = ''
        occurrence = {}
    
        for i in range(int(token_count)):
            input_ids = self._pipeline.encode(ctx)
            input_ids = input_ids[-self.CTX_LIMIT:] if i == 0 else [token]
            
            out, self.state = self._model.forward(input_ids, self.state)
    
            for n in occurrence:
                out[n] -= (presencePenalty + occurrence[n] * countPenalty)
    
            token = self._pipeline.sample_logits(
                out,
                temperature=temperature,
                top_p=top_p)
            if token in [261]:
                break
    
            all_tokens.append(token)
    
            for xxx in occurrence:
                occurrence[xxx] *= self.PENALTY_DECAY
    
            ttt = self._pipeline.decode([token])
            www = 1
            if ttt in ' \t0123456789':
                www = 0
            if token not in occurrence:
                occurrence[token] = www
            else:
                occurrence[token] += www
    
            tmp = self._pipeline.decode(all_tokens[out_last:])
            if '\ufffd' not in tmp:
                out_str += tmp
                out_last = i + 1

                if out_str.strip().endswith("\n\n"):
                    break
    
        # Cleanup and timing
        del out
        gc.collect()
        torch.cuda.empty_cache()
        
        return out_str.strip()

In [37]:
model = RWKVModel(model=rwkv_model, pipeline=rwkv_pipeline)

In [38]:
model.reset_state()
ans = model.generate('User: Hello, what is your favourite city?\n\nAssistant:<think')

Input prompt:  User: Hello, what is your favourite city?

Assistant:<think
Output:  >Hmm, the user is asking about my favorite city. This is a straightforward question but requires a thoughtful response since it's subjective and depends on context. 
I should acknowledge that cities are complex and multifaceted, so I can't pick just one. Instead, I'll highlight key aspects of different cities that make them special—like cultural richness, historical significance, or unique experiences. 
I'll structure the response by mentioning three distinct cities with their own strengths: Paris for art and romance, Tokyo for innovation and efficiency, and New York for energy and diversity. Each point should be concise but vivid enough to paint a picture without being overwhelming. 
The tone should be enthusiastic but balanced, avoiding clichés while still conveying genuine appreciation. I'll end by inviting the user to share their own favorite city to keep the conversation open.</think>That's a wonde